# Filesystem Artifacts Demo

This notebook shows how to use `Policy.artifacts` to expose datasets, skills, and experiment traces as read-only files in the agent's working directory under `_vero/`.

The agent can read these files but cannot modify them — enforced by workspace access rules.

In [ ]:
import subprocess
import tempfile
from pathlib import Path

from datasets import Dataset, DatasetDict

## Setup: Create a git repo, dataset, and skills directory

In [ ]:
# Create a temporary project directory
tmp = Path(tempfile.mkdtemp())
repo = tmp / "project"
repo.mkdir()

# Initialize git repo
for cmd in [
    ["git", "init"],
    ["git", "config", "user.name", "demo"],
    ["git", "config", "user.email", "demo@example.com"],
]:
    subprocess.run(cmd, cwd=repo, capture_output=True, check=True)

(repo / "main.py").write_text("print('hello world')\n")
subprocess.run(["git", "add", "."], cwd=repo, capture_output=True, check=True)
subprocess.run(["git", "commit", "-m", "init"], cwd=repo, capture_output=True, check=True)
subprocess.run(["git", "branch", "-M", "main"], cwd=repo, capture_output=True, check=True)

print(f"Project: {repo}")

In [ ]:
# Create a dataset with train/validation/test splits
ds = DatasetDict({
    "train": Dataset.from_dict({
        "question": ["What is 2+2?", "What is 3*3?", "What is 10/2?"],
        "answer": ["4", "9", "5"],
        "difficulty": ["easy", "easy", "easy"],
    }),
    "validation": Dataset.from_dict({
        "question": ["What is 7*8?", "What is 144/12?"],
        "answer": ["56", "12"],
        "difficulty": ["medium", "medium"],
    }),
    "test": Dataset.from_dict({
        "question": ["What is 17*23?"],
        "answer": ["391"],
        "difficulty": ["hard"],
    }),
})
ds_dir = tmp / "dataset"
ds.save_to_disk(str(ds_dir))
print(f"Dataset: {ds_dir}")
print(f"Splits: {list(ds.keys())}")

In [ ]:
# Create skills (markdown cookbooks the agent can reference)
skills_dir = tmp / "cookbooks"
skills_dir.mkdir()

(skills_dir / "prompt_engineering.md").write_text("""
# Prompt Engineering Cookbook

## Chain of Thought
Ask the model to think step by step before answering.

## Few-Shot Examples
Include 2-3 examples in the prompt to establish the expected format.
""")

(skills_dir / "tool_design.md").write_text("""
# Tool Design Cookbook

## Calculator Tool
For math tasks, provide a calculator tool that evaluates expressions.
""")

print(f"Skills: {skills_dir}")
print(f"Files: {[f.name for f in skills_dir.iterdir()]}")

## Configure artifacts on Policy

Each artifact type is a self-contained object that knows how to materialize itself:
- `DatasetArtifact()` — writes per-sample JSON files for viewable splits
- `SkillsArtifact()` — copies skill directories by namespace
- `TracesArtifact()` — writes experiment traces after each evaluation

In [ ]:
from vero.agents.vero import VeroAgent
from vero.artifacts import DatasetArtifact, SkillsArtifact, TracesArtifact
from vero.policy import Policy

# Use a temp dir for vero home (sessions, datasets)
vero_home = tmp / "vero_home"
vero_home.mkdir()

agent = VeroAgent(tool_sets=[])
policy = Policy(
    project_path=repo,
    dataset=ds_dir,
    agent=agent,
    task="main",
    vero_home=vero_home,
    use_copy=False,
    skills={"agent-cookbooks": skills_dir},
    artifacts=[
        DatasetArtifact(),    # viewable splits as JSON
        SkillsArtifact(),     # cookbook markdown files
        TracesArtifact(),     # experiment traces (after evals)
    ],
    train_budget=3,
    validation_budget=2,
)
await policy.init()
print(f"Session: {policy.session_id}")

## Inspect the materialized filesystem

After `init()`, the `_vero/` directory is populated with artifacts.
The agent sees this as part of its working directory.

In [ ]:
vero_dir = Path(policy.session.workspace.project_path) / "_vero"

print("=== _vero/ directory structure ===")
for p in sorted(vero_dir.rglob("*")):
    rel = p.relative_to(vero_dir)
    indent = "  " * (len(rel.parts) - 1)
    if p.is_dir():
        print(f"{indent}{rel.name}/")
    else:
        print(f"{indent}{rel.name} ({p.stat().st_size} bytes)")

In [ ]:
import json

# Dataset: only viewable splits are materialized
print("=== Materialized dataset splits ===")
datasets_dir = vero_dir / "datasets"
if datasets_dir.exists():
    for split_dir in sorted(datasets_dir.rglob("*")):
        if split_dir.is_dir() and split_dir.parent != datasets_dir:
            samples = list(split_dir.glob("*.json"))
            print(f"  {split_dir.relative_to(datasets_dir)}: {len(samples)} samples")
            if samples:
                sample = json.loads(samples[0].read_text())
                print(f"    Sample 0: {sample}")

In [ ]:
# Skills: copied by namespace
print("=== Materialized skills ===")
skills_out = vero_dir / "skills"
if skills_out.exists():
    for ns_dir in sorted(skills_out.iterdir()):
        if ns_dir.is_dir():
            files = list(ns_dir.glob("*"))
            print(f"  Namespace '{ns_dir.name}': {[f.name for f in files]}")

## Workspace access control

The `_vero/` directory is automatically added with READ-only access.
The agent can read these files but cannot write to them.

In [ ]:
workspace = policy.session.workspace
print("=== Workspace access rules ===")
print(f"Default access: {workspace.default_access}")
for rule in workspace.accesses:
    print(f"  {rule.pattern}: {rule.access_type}")

print()

# Verify access (paths relative to project_path)
print("Can read _vero/datasets/dataset/train/0.json?", workspace.can_read("_vero/datasets/dataset/train/0.json"))
print("Can write _vero/datasets/dataset/train/0.json?", workspace.can_write("_vero/datasets/dataset/train/0.json"))
print("Can write main.py?", workspace.can_write("main.py"))

## Custom artifacts

You can create your own artifact types by subclassing `FileSystemArtifact`.

In [ ]:
from dataclasses import dataclass

from vero.artifacts import FileSystemArtifact


@dataclass
class ReadmeArtifact(FileSystemArtifact):
    """Writes a README.md to _vero/ with project context."""

    content: str = "# Agent Workspace\n\nThis directory contains read-only artifacts."

    async def on_init(self, policy, dest, sandbox):
        await sandbox.write_file(f"{dest}/README.md", self.content)

    async def on_experiment(self, policy, experiment, dest, sandbox):
        pass  # Nothing to do after experiments


# Usage:
# Policy(artifacts=[DatasetArtifact(), ReadmeArtifact(content="Custom readme")])
print("Custom artifact defined!")

In [ ]:
# Cleanup
policy.finish()
print("Done!")